# Eye State Detection for Driver Drowsiness — Final Project Notebook

Everything this notebook produces is saved in one Drive folder: `eye_project/Eye_State_Detection_Final/`
(`models/`, `results/`, `config/`, `logs/`, `README_results.md`).

| Model | Training data | Augmentation | Purpose |
|---|---|---|---|
| v1 | our dataset | crop, flip, rotation, brightness/contrast | first model |
| v2 | our dataset | v1 + blur + low-resolution simulation | robustness to blurry, small eyes |
| CEW-only | CEW train split | as v2 | domain-shift experiment |
| v3 | our dataset + CEW train split | as v2 | more varied faces |
| v4 | our dataset + CEW train split + FL3D train drivers (real night driving, NITYMED frames) | as v2 | real driving environment |

The final model is chosen on **validation** data only; every test set is used once, for reporting.

## 0. Settings and data

In [ ]:
OLD        = "/content/drive/MyDrive/eye_project"
FINAL      = f"{OLD}/Eye_State_Detection_Final"
DATA_ZIP   = f"{OLD}/my_dataset/Mydataset.zip"
CEW_ZIP    = f"{OLD}/my_dataset/dataset_B_Facial_Images.zip"
OLD_SPLIT  = f"{OLD}/split.csv"
OLD_FACE   = f"{OLD}/models/face_landmarker.task"

MODELS, RESULTS, CONFIG, LOGS = (f"{FINAL}/{d}" for d in ["models", "results", "config", "logs"])
FACE_MODEL = f"{MODELS}/face_landmarker.task"
SPLIT_FILE = f"{FINAL}/split.csv"

FL3D_SLUG = "matjazmuc/frame-level-driver-drowsiness-detection-fl3d"
NITYMED_SLUG = "nikospetrellis/nitymed"
FL3D_MAX_PER_CLASS = 4000
RUN_VIDEO_TEST = True
VIDEO_TEST_COUNT = 5

RETRAIN = False
EPOCHS = 15
SEED = 0

In [ ]:
!pip install -q mediapipe kagglehub tabulate

from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o "{DATA_ZIP}" -d /content/my_dataset
!unzip -q -o "{CEW_ZIP}" -d /content/cew100

import os, shutil, urllib.request
for d in [MODELS, RESULTS, CONFIG, LOGS]:
    os.makedirs(d, exist_ok=True)
if not os.path.exists(FACE_MODEL):
    if os.path.exists(OLD_FACE):
        shutil.copy(OLD_FACE, FACE_MODEL)
    else:
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", FACE_MODEL)
if not os.path.exists(SPLIT_FILE):
    shutil.copy(OLD_SPLIT, SPLIT_FILE)

FL3D_DIR, NITYMED_DIR = None, None
try:
    import kagglehub
    FL3D_DIR = kagglehub.dataset_download(FL3D_SLUG)
    if RUN_VIDEO_TEST:
        NITYMED_DIR = kagglehub.dataset_download(NITYMED_SLUG)
except Exception as e:
    print("Kaggle download failed:", e)
    print("Add your Kaggle username/key (Colab: key icon -> Secrets: KAGGLE_USERNAME, KAGGLE_KEY) and run this cell again.")
print("FL3D:", FL3D_DIR, "| NITYMED:", NITYMED_DIR)

In [ ]:
import csv, json, math, random, re, time
from collections import Counter, defaultdict, deque
from math import atan2, degrees, hypot
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

device = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["closed", "open"]
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
print("Device:", device)

## 1. Our dataset and the person-level split

In [ ]:
split_df = pd.read_csv(SPLIT_FILE)
missing = (~split_df.path.map(os.path.exists)).sum()
print(f"{len(split_df)} images in split.csv, missing on disk: {missing}")
table = split_df.groupby("split").agg(images=("label", "size"), closed=("label", lambda s: (s == 0).sum()),
                                      open=("label", lambda s: (s == 1).sum())).loc[["train", "val", "test"]]
display(table)
table.to_csv(f"{RESULTS}/our_split_table.csv")

def fingerprint(path):
    a = np.asarray(Image.open(path).convert("L").resize((16, 16)), dtype=np.float32)
    return (a > a.mean()).tobytes()

prints = {p: fingerprint(p) for p in split_df.path}
train_prints = {prints[p] for p in split_df[split_df.split == "train"].path}
leak = {s: int(sum(prints[p] in train_prints for p in split_df[split_df.split == s].path)) for s in ["val", "test"]}
print("Near-duplicates of training images:", leak)

## 2. Training and evaluation code

In [ ]:
IMG_SIZE = 112
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class RandomLowRes:
    def __init__(self, p=0.5, min_size=20, max_size=56):
        self.p, self.min_size, self.max_size = p, min_size, max_size
    def __call__(self, img):
        if random.random() < self.p:
            s = random.randint(self.min_size, self.max_size)
            w, h = img.size
            img = img.resize((s, s), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return img

aug_basic = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
aug_full = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    RandomLowRes(p=0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.5, contrast=0.5),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class OurDataset(Dataset):
    def __init__(self, split, transform):
        rows = split_df[split_df.split == split]
        self.items = list(zip(rows.path, rows.label.astype(int)))
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.transform(Image.open(path).convert("RGB")), label

class FaceCrops(Dataset):
    def __init__(self, faces, ids, transform):
        self.samples = [(crop, faces[i][0]) for i in ids for crop in faces[i][1]]
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, k):
        crop, label = self.samples[k]
        return self.transform(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))), label

def new_model(pretrained=True):
    m = models.mobilenet_v3_small(weights="IMAGENET1K_V1" if pretrained else None)
    m.classifier[3] = nn.Linear(1024, 2)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.momentum = 0.1
    return m.to(device)

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    criterion = nn.CrossEntropyLoss()
    total, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            seen += y.size(0)
    return total / seen, correct / seen

def get_model(name, train_set, val_set):
    path = f"{MODELS}/{name}.pt"
    hist_path = f"{RESULTS}/history_{name}.json"
    if Path(path).exists() and not RETRAIN:
        m = new_model(False)
        m.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded {path}")
        return m.eval()
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    m = new_model(True)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    tl_ = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
    vl_ = DataLoader(val_set, batch_size=64, num_workers=2)
    hist, best = [], -1.0
    print(f"Training {name}: {len(train_set)} train / {len(val_set)} val eye images")
    for ep in range(1, EPOCHS + 1):
        tl, ta = run_epoch(m, tl_, opt)
        vl, va = run_epoch(m, vl_)
        sched.step()
        hist.append({"epoch": ep, "train_loss": tl, "train_acc": ta, "val_loss": vl, "val_acc": va})
        mark = ""
        if va > best:
            best = va; torch.save(m.state_dict(), path); mark = "  <- best"
        print(f"  epoch {ep:2d} | train {tl:.4f} / {ta:.4f} | val {vl:.4f} / {va:.4f}{mark}")
    json.dump(hist, open(hist_path, "w"), indent=1)
    h = pd.DataFrame(hist)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(h.epoch, h.train_loss, label="train"); a1.plot(h.epoch, h.val_loss, label="validation"); a1.set_title("Loss")
    a2.plot(h.epoch, h.train_acc, label="train"); a2.plot(h.epoch, h.val_acc, label="validation"); a2.set_title("Accuracy")
    for a in (a1, a2):
        a.set_xlabel("epoch"); a.legend()
    plt.suptitle(f"Training curves: {name}"); plt.tight_layout()
    plt.savefig(f"{RESULTS}/training_curves_{name}.png", dpi=150, bbox_inches="tight"); plt.show()
    m.load_state_dict(torch.load(path, map_location=device))
    return m.eval()

def probs_images(model, dataset):
    model.eval()
    out = []
    with torch.no_grad():
        for x, _ in DataLoader(dataset, batch_size=128, num_workers=2):
            out.append(torch.softmax(model(x.to(device)), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

def probs_faces(model, faces, ids):
    model.eval()
    x = torch.stack([eval_tf(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for i in ids for c in faces[i][1]])
    out = []
    with torch.no_grad():
        for k in range(0, len(x), 128):
            out.append(torch.softmax(model(x[k:k + 128].to(device)), 1)[:, 1].cpu())
    return torch.cat(out).numpy().reshape(-1, 2).mean(1)

def scores(y, p, t=0.5):
    pred = (p > t).astype(int)
    r = {"n": int(len(y)), "accuracy": float((pred == y).mean()),
         "closed_recall": float(((pred == 0) & (y == 0)).sum() / max(1, (y == 0).sum())),
         "open_recall": float(((pred == 1) & (y == 1)).sum() / max(1, (y == 1).sum()))}
    r["balanced_accuracy"] = (r["closed_recall"] + r["open_recall"]) / 2
    r["auc"] = float(roc_auc_score(y, p)) if len(set(y)) == 2 else float("nan")
    return r

## 3. Face pipeline: MediaPipe eye crops and EAR

In [ ]:
CROP_SIZE, CROP_SCALE = 128, 2.2
EYE_CORNERS = [(33, 133), (362, 263)]
RIGHT_EAR = [33, 160, 158, 133, 153, 144]
LEFT_EAR = [362, 385, 387, 263, 373, 380]

landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_buffer=open(FACE_MODEL, "rb").read()),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))

def eye_crops(frame, lm):
    h, w = frame.shape[:2]
    out = []
    for i1, i2 in EYE_CORNERS:
        p1 = np.array([lm[i1].x * w, lm[i1].y * h]); p2 = np.array([lm[i2].x * w, lm[i2].y * h])
        if p1[0] > p2[0]:
            p1, p2 = p2, p1
        c = (p1 + p2) / 2
        side = max(hypot(*(p2 - p1)) * CROP_SCALE, 1.0)
        M = cv2.getRotationMatrix2D((float(c[0]), float(c[1])), degrees(atan2(p2[1] - p1[1], p2[0] - p1[0])), CROP_SIZE / side)
        M[0, 2] += CROP_SIZE / 2 - c[0]; M[1, 2] += CROP_SIZE / 2 - c[1]
        out.append(cv2.warpAffine(frame, M, (CROP_SIZE, CROP_SIZE), borderMode=cv2.BORDER_REPLICATE))
    return out

def ear_of(lm, w, h):
    def one(idx):
        p = [np.array([lm[i].x * w, lm[i].y * h]) for i in idx]
        return (np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])) / (2 * np.linalg.norm(p[0] - p[3]) + 1e-9)
    return (one(RIGHT_EAR) + one(LEFT_EAR)) / 2

def face_item(frame, label, pad_small=False):
    if pad_small:
        pad = int(0.4 * max(frame.shape[:2]))
        frame = cv2.copyMakeBorder(frame, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(128, 128, 128))
        if max(frame.shape[:2]) < 400:
            s = 400 / max(frame.shape[:2])
            frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    elif max(frame.shape[:2]) > 1280:
        s = 1280 / max(frame.shape[:2])
        frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    if not res.face_landmarks:
        return None
    lm = res.face_landmarks[0]
    h, w = frame.shape[:2]
    return (label, eye_crops(frame, lm), ear_of(lm, w, h))

## 4. CEW (Closed Eyes in the Wild): 70% train / 30% test

In [ ]:
cew_dirs = {0: next(Path("/content/cew100").rglob("ClosedFace")), 1: next(Path("/content/cew100").rglob("OpenFace"))}
cew, cew_no_face = [], 0
for label, folder in cew_dirs.items():
    for path in sorted(folder.glob("*")):
        img = cv2.imread(str(path))
        item = face_item(img, label, pad_small=True) if img is not None else None
        if item is None:
            cew_no_face += 1
        else:
            cew.append(item)
cew_y = np.array([c[0] for c in cew]); cew_ear = np.array([c[2] for c in cew])
idx = np.arange(len(cew))
cew_train, cew_test = train_test_split(idx, test_size=0.30, stratify=cew_y, random_state=0)
cew_fit, cew_val = train_test_split(cew_train, test_size=0.15, stratify=cew_y[cew_train], random_state=0)
print(f"CEW faces: {len(cew)} (no face: {cew_no_face}) | train {len(cew_fit)}, val {len(cew_val)}, test {len(cew_test)}")

## 5. FL3D: real night-time driving frames (from the NITYMED videos)

Labels come from the dataset's `annotations_*.json`: `alert` → open, `microsleep` → closed; `yawning` frames are left out.
(FL3D already removed blink frames, so `alert` frames have open eyes.)
Frames are split **by video** (one folder per video): no video appears in two splits.
The NITYMED system test in part 12 only uses FL3D **test** videos.

In [ ]:
STATE = {"alert": 1, "microsleep": 0, "yawning": -1}

def find_state(v):
    if isinstance(v, str):
        s = v.lower().strip()
        return STATE.get(s)
    if isinstance(v, dict):
        for k in ["driver_state", "state", "label", "class", "category"]:
            if k in v and isinstance(v[k], str) and v[k].lower().strip() in STATE:
                return STATE[v[k].lower().strip()]
        for x in v.values():
            r = find_state(x) if isinstance(x, (str, dict)) else None
            if r is not None:
                return r
    return None

def find_path(d):
    for x in d.values():
        if isinstance(x, str) and Path(x).suffix.lower() in EXTS:
            return x
    return None

fl3d_paths, fl3d_labels = [], []
if FL3D_DIR:
    root = Path(FL3D_DIR)
    imgs = [p for p in root.rglob("*") if p.suffix.lower() in EXTS]
    by_two = {(p.parent.name, p.name): p for p in imgs}
    by_name = defaultdict(list)
    for p in imgs:
        by_name[p.name].append(p)
    ann_files = sorted(root.rglob("annotations_*.json"))
    ann_files = [a for a in ann_files if "hold" not in a.name] or ann_files
    print("Annotation files:", [a.name for a in ann_files])
    labels = {}
    for a in ann_files:
        data = json.load(open(a))
        entries = data.items() if isinstance(data, dict) else [(find_path(d), d) for d in data if isinstance(d, dict)]
        for key, val in entries:
            if key is None:
                continue
            lab = find_state(val)
            if lab is None:
                continue
            k = Path(str(key))
            p = by_two.get((k.parent.name, k.name)) or (by_name[k.name][0] if len(by_name[k.name]) == 1 else None)
            if p is not None:
                labels[p] = lab
    if ann_files and not labels:
        data = json.load(open(ann_files[0]))
        sample = list(data.items())[:2] if isinstance(data, dict) else data[:2]
        print("Could not read labels. Example entries:", str(sample)[:800])
    fl3d_paths, fl3d_labels = list(labels), list(labels.values())
    print(f"FL3D images: {len(imgs)}, labelled: {len(labels)}, counts: {Counter(fl3d_labels)} (1 alert, 0 microsleep, -1 yawning)")

fl3d, fl3d_groups = [], []
if fl3d_paths:
    rng = random.Random(SEED)
    chosen = []
    for lab in [0, 1]:
        ps = sorted(p for p, l in zip(fl3d_paths, fl3d_labels) if l == lab)
        rng.shuffle(ps)
        chosen += [(p, lab) for p in ps[:FL3D_MAX_PER_CLASS]]
    no_face = 0
    for p, lab in chosen:
        img = cv2.imread(str(p))
        item = face_item(img, lab) if img is not None else None
        if item is None:
            no_face += 1
        else:
            fl3d.append(item); fl3d_groups.append(p.parent.name)
    print(f"FL3D faces used: {len(fl3d)} (no face: {no_face}), videos: {len(set(fl3d_groups))}")

fl3d_fit = fl3d_val = fl3d_test = np.array([], dtype=int)
FL3D_TEST_VIDEOS = set()
if fl3d:
    fl3d_y = np.array([f[0] for f in fl3d]); fl3d_ear = np.array([f[2] for f in fl3d])
    g = np.array(fl3d_groups); idx = np.arange(len(fl3d))
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(idx, groups=g))
    fi, va = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(tr, groups=g[tr]))
    fl3d_fit, fl3d_val, fl3d_test = tr[fi], tr[va], te
    FL3D_TEST_VIDEOS = set(g[fl3d_test])
    print(f"FL3D split by video: train {len(fl3d_fit)} ({len(set(g[fl3d_fit]))} videos), "
          f"val {len(fl3d_val)} ({len(set(g[fl3d_val]))} videos), test {len(fl3d_test)} ({len(FL3D_TEST_VIDEOS)} videos)")
    for name, ids in [("train", fl3d_fit), ("val", fl3d_val), ("test", fl3d_test)]:
        print(f"  {name}: closed {(fl3d_y[ids] == 0).sum()}, open {(fl3d_y[ids] == 1).sum()}")
    print("Test videos:", sorted(map(str, FL3D_TEST_VIDEOS)))

## 6. Train the five models

In [ ]:
OT, OV = OurDataset("train", aug_full), OurDataset("val", eval_tf)
models_ = {}
models_["v1"] = get_model("v1", OurDataset("train", aug_basic), OV)
models_["v2"] = get_model("v2", OT, OV)
models_["cew_only"] = get_model("cew_only", FaceCrops(cew, cew_fit, aug_full), FaceCrops(cew, cew_val, eval_tf))
models_["v3"] = get_model("v3", ConcatDataset([OT, FaceCrops(cew, cew_fit, aug_full)]),
                          ConcatDataset([OV, FaceCrops(cew, cew_val, eval_tf)]))
if fl3d:
    models_["v4"] = get_model("v4", ConcatDataset([OT, FaceCrops(cew, cew_fit, aug_full), FaceCrops(fl3d, fl3d_fit, aug_full)]),
                              ConcatDataset([OV, FaceCrops(cew, cew_val, eval_tf), FaceCrops(fl3d, fl3d_val, eval_tf)]))
print("Models:", list(models_))

## 7. Choose the final model on validation data

In [ ]:
y_our_val = np.array([l for _, l in OurDataset("val", eval_tf).items])
val_rows = []
for name, m in models_.items():
    r = {"model": name,
         "our_val": scores(y_our_val, probs_images(m, OurDataset("val", eval_tf)))["accuracy"],
         "cew_val": scores(cew_y[cew_val], probs_faces(m, cew, cew_val))["accuracy"]}
    if fl3d:
        r["fl3d_val"] = scores(fl3d_y[fl3d_val], probs_faces(m, fl3d, fl3d_val))["accuracy"]
    r["mean_val"] = float(np.mean([v for k, v in r.items() if k.endswith("_val")]))
    val_rows.append(r)
val_table = pd.DataFrame(val_rows).set_index("model")
display(val_table.style.format("{:.2%}"))
val_table.to_csv(f"{RESULTS}/validation_table.csv")
FINAL_NAME = val_table.mean_val.idxmax()
final_model = models_[FINAL_NAME]
shutil.copy(f"{MODELS}/{FINAL_NAME}.pt", f"{MODELS}/eye_state_final.pt")
print("Final model (best mean validation accuracy):", FINAL_NAME)

## 8. Decision threshold and hysteresis from validation data

Balanced accuracy of the final model on all validation data, for every threshold.
The decision threshold is the best one; the hysteresis band is the range of thresholds whose balanced
accuracy stays within 0.5 percentage points of the best (at least ±0.02, at most ±0.10), so switching inside the band
costs almost no accuracy but stops flicker.

In [ ]:
pv = [probs_images(final_model, OurDataset("val", eval_tf)), probs_faces(final_model, cew, cew_val)]
yv = [y_our_val, cew_y[cew_val]]
if fl3d:
    pv.append(probs_faces(final_model, fl3d, fl3d_val)); yv.append(fl3d_y[fl3d_val])
pv, yv = np.concatenate(pv), np.concatenate(yv)

grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
bal = np.array([scores(yv, pv, t)["balanced_accuracy"] for t in grid])
T = float(grid[bal.argmax()])
plateau = grid[bal >= bal.max() - 0.005]
T_CLOSE = round(float(max(min(plateau.min(), T - 0.02), T - 0.10, 0.01)), 2)
T_OPEN = round(float(min(max(plateau.max(), T + 0.02), T + 0.10, 0.99)), 2)
print(f"threshold {T:.2f} (balanced acc {bal.max():.4f}), hysteresis close < {T_CLOSE:.2f}, open > {T_OPEN:.2f}")

plt.figure(figsize=(8, 4))
plt.plot(grid, bal)
plt.axvline(T, color="k", label=f"threshold {T:.2f}")
plt.axvspan(T_CLOSE, T_OPEN, alpha=0.2, label="hysteresis band")
plt.xlabel("threshold on P(open)"); plt.ylabel("balanced accuracy (validation)"); plt.legend()
plt.title(f"Threshold selection, model {FINAL_NAME}")
plt.savefig(f"{RESULTS}/threshold_selection.png", dpi=150, bbox_inches="tight"); plt.show()

demo_config = {
    "model_file": "models/eye_state_final.pt", "face_model_file": "models/face_landmarker.task",
    "final_model": FINAL_NAME, "camera_index": 0, "img_size": IMG_SIZE, "crop_size": CROP_SIZE, "crop_scale": CROP_SCALE,
    "threshold": T, "threshold_close": T_CLOSE, "threshold_open": T_OPEN,
    "smoothing_tau_s": 0.10, "blink_max_s": 0.5, "microsleep_max_s": 1.5,
    "no_face_alarm_s": 2.0, "perclos_window_s": 60.0, "perclos_warning": 0.15,
}
json.dump(demo_config, open(f"{CONFIG}/demo_config.json", "w"), indent=2)
print(json.dumps(demo_config, indent=2))

## 9. Test results (every test set used once)

In [ ]:
our_test = OurDataset("test", eval_tf)
y_our = np.array([l for _, l in our_test.items])
tests = {"our_test": None, "cew_test": (cew, cew_test, cew_y)}
if fl3d:
    tests["fl3d_test"] = (fl3d, fl3d_test, fl3d_y)

rows, probs_store = [], {}
for name, m in models_.items():
    for tname, spec in tests.items():
        if spec is None:
            p, y = probs_images(m, our_test), y_our
        else:
            faces, ids, yy = spec
            p, y = probs_faces(m, faces, ids), yy[ids]
        probs_store[(name, tname)] = (p, y)
        t = T if name == FINAL_NAME else 0.5
        rows.append({"model": name, "test_set": tname, "threshold": t, **scores(y, p, t)})
for tname, ears, ids, yy, train_ids in [("cew_test", cew_ear, cew_test, cew_y, cew_train)] + \
        ([("fl3d_test", fl3d_ear, fl3d_test, fl3d_y, np.concatenate([fl3d_fit, fl3d_val]))] if fl3d else []):
    th = np.arange(0.05, 0.45, 0.005)
    t_ear = float(max(th, key=lambda t: ((ears[train_ids] > t).astype(int) == yy[train_ids]).mean()))
    for label, t in [("EAR_0.20", 0.20), ("EAR_tuned", t_ear)]:
        rows.append({"model": label, "test_set": tname, "threshold": t, **scores(yy[ids], ears[ids], t)})

results = pd.DataFrame(rows)
results.to_csv(f"{RESULTS}/test_results_long.csv", index=False)
acc = results.pivot_table(index="model", columns="test_set", values="accuracy")
order = [m for m in ["v1", "v2", "cew_only", "v3", "v4", "EAR_0.20", "EAR_tuned"] if m in acc.index]
acc = acc.loc[order]
acc.to_csv(f"{RESULTS}/test_accuracy_table.csv")
display(acc.style.format("{:.2%}", na_rep="needs full face"))
display(results[results.model == FINAL_NAME].set_index("test_set"))

In [ ]:
for name in models_:
    fig, axes = plt.subplots(1, len(tests), figsize=(5.5 * len(tests), 4.6))
    for ax, tname in zip(np.atleast_1d(axes), tests):
        p, y = probs_store[(name, tname)]
        t = T if name == FINAL_NAME else 0.5
        ConfusionMatrixDisplay(confusion_matrix(y, (p > t).astype(int), labels=[0, 1]), display_labels=CLASSES).plot(
            ax=ax, cmap="Blues", values_format="d", colorbar=False)
        ax.set_title(f"{name} | {tname} | acc {((p > t).astype(int) == y).mean():.2%}")
    plt.tight_layout()
    plt.savefig(f"{RESULTS}/confusion_matrices_{name}.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2 if fl3d else 1, figsize=(14 if fl3d else 7, 4))
for ax, (tname, ears, yy) in zip(np.atleast_1d(axes), [("CEW", cew_ear, cew_y)] + ([("FL3D", fl3d_ear, fl3d_y)] if fl3d else [])):
    ax.hist(ears[yy == 0], bins=50, alpha=0.6, label="closed"); ax.hist(ears[yy == 1], bins=50, alpha=0.6, label="open")
    ax.axvline(0.20, color="k", linestyle="--", label="0.20"); ax.set_xlabel("EAR"); ax.set_title(f"EAR on {tname}"); ax.legend()
plt.savefig(f"{RESULTS}/ear_histograms.png", dpi=150, bbox_inches="tight"); plt.show()

## 10. Errors of the final model

In [ ]:
for tname in tests:
    p, y = probs_store[(FINAL_NAME, tname)]
    wrong = np.where((p > T).astype(int) != y)[0][:8]
    if len(wrong) == 0:
        print(tname, ": no errors"); continue
    fig, axes = plt.subplots(1, len(wrong), figsize=(2.4 * len(wrong), 2.8), squeeze=False)
    for ax, i in zip(axes[0], wrong):
        if tname == "our_test":
            img = Image.open(our_test.items[i][0]).convert("RGB")
        else:
            faces, ids, _ = tests[tname]
            img = cv2.cvtColor(np.hstack(faces[ids[i]][1]), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(f"true {CLASSES[y[i]]}\nP(open) {p[i]:.2f}", fontsize=8); ax.axis("off")
    plt.suptitle(f"{FINAL_NAME} errors on {tname}")
    plt.savefig(f"{RESULTS}/errors_{tname}.png", dpi=150, bbox_inches="tight"); plt.show()

## 11. Speed and size

In [ ]:
def ms_per_frame(model, dev, runs=100):
    model = model.to(dev).eval()
    x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=dev)
    with torch.no_grad():
        for _ in range(10):
            model(x)
        if dev == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            model(x)
        if dev == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000

speed = {"final_model": FINAL_NAME, "params_M": sum(p.numel() for p in final_model.parameters()) / 1e6,
         "file_MB": Path(f"{MODELS}/eye_state_final.pt").stat().st_size / 1e6,
         "colab_cpu_ms_per_frame": ms_per_frame(final_model, "cpu")}
final_model.to(device)
rows = []
for name, m in [("MobileNetV3-Small", models.mobilenet_v3_small()), ("MobileNetV3-Large", models.mobilenet_v3_large()),
                ("EfficientNet-B0", models.efficientnet_b0()), ("ResNet-18", models.resnet18()), ("ResNet-50", models.resnet50())]:
    t = ms_per_frame(m, "cpu", 50)
    rows.append({"model": name, "params_M": round(sum(q.numel() for q in m.parameters()) / 1e6, 2), "cpu_ms_per_frame": round(t, 1)})
speed_table = pd.DataFrame(rows).set_index("model")
speed_table.to_csv(f"{RESULTS}/speed_comparison.csv")
json.dump(speed, open(f"{RESULTS}/speed_final_model.json", "w"), indent=2)
print(speed); display(speed_table)

## 12. System test on real driving videos (NITYMED microsleep videos)

The same frame-rate-independent logic as `demo.py` (smoothing with time constant, hysteresis,
blink / microsleep / long-closure events, PERCLOS, no-face alarm) runs on each video.
A per-frame log (CSV) and an event summary (JSON) are saved for each video in `logs/`.

In [ ]:
class DrowsinessTracker:
    def __init__(self, cfg):
        self.cfg = cfg
        self.smoothed, self.closed, self.closed_since, self.last_t, self.face_lost_since = None, False, None, None, None
        self.history = deque()
        self.counts = {"blink": 0, "microsleep": 0, "long_closure": 0, "no_face_alarm": 0}
        self.events, self.no_face_alarm_on = [], False
    def perclos(self):
        total = sum(d for _, d, _ in self.history)
        return sum(d for _, d, c in self.history if c) / total if total > 0 else 0.0
    def _event(self, kind, t, dur):
        self.counts[kind] += 1
        self.events.append({"event": kind, "time_s": round(t, 3), "duration_s": round(dur, 3)})
        return kind
    def _end_closure(self, t):
        d = t - self.closed_since
        kind = "blink" if d < self.cfg["blink_max_s"] else "microsleep" if d < self.cfg["microsleep_max_s"] else "long_closure"
        self.closed, self.closed_since = False, None
        return self._event(kind, t, d)
    def update(self, t, p):
        dt = 0.0 if self.last_t is None else max(0.0, t - self.last_t)
        self.last_t, event = t, ""
        if p is None:
            if self.face_lost_since is None:
                self.face_lost_since = t
                if self.closed:
                    self.events.append({"event": "closure_interrupted_no_face", "time_s": round(t, 3), "duration_s": round(t - self.closed_since, 3)})
                self.closed, self.closed_since, self.smoothed = False, None, None
            lost = t - self.face_lost_since
            if lost >= self.cfg["no_face_alarm_s"] and not self.no_face_alarm_on:
                self.no_face_alarm_on = True
                event = self._event("no_face_alarm", t, lost)
            return "NO_FACE", event, 0.0, self.no_face_alarm_on
        self.face_lost_since, self.no_face_alarm_on = None, False
        if self.smoothed is None:
            self.smoothed = p
        else:
            self.smoothed += (1 - math.exp(-dt / self.cfg["smoothing_tau_s"]) if dt > 0 else 0.0) * (p - self.smoothed)
        if dt > 0:
            self.history.append((t, dt, self.closed))
        while self.history and self.history[0][0] < t - self.cfg["perclos_window_s"]:
            self.history.popleft()
        if not self.closed and self.smoothed < self.cfg["threshold_close"]:
            self.closed, self.closed_since = True, t
        elif self.closed and self.smoothed > self.cfg["threshold_open"]:
            event = self._end_closure(t)
        cl = t - self.closed_since if self.closed else 0.0
        return ("CLOSED" if self.closed else "OPEN"), event, cl, self.closed and cl >= self.cfg["microsleep_max_s"]
    def finish(self, t):
        if self.closed:
            self._end_closure(t)

def run_video(path, cfg, model):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    lmk = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_buffer=open(FACE_MODEL, "rb").read()),
        running_mode=vision.RunningMode.VIDEO, num_faces=1))
    tr, rows, i = DrowsinessTracker(cfg), [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = i / fps
        if max(frame.shape[:2]) > 1280:
            s = 1280 / max(frame.shape[:2]); frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        res = lmk.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), int(t * 1000))
        p = ear = None
        if res.face_landmarks:
            lm = res.face_landmarks[0]; h, w = frame.shape[:2]
            x = torch.stack([eval_tf(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in eye_crops(frame, lm)]).to(device)
            with torch.no_grad():
                p = float(torch.softmax(model(x), 1)[:, 1].mean())
            ear = ear_of(lm, w, h)
        state, ev, cl, alarm = tr.update(t, p)
        rows.append({"t_s": round(t, 3), "frame": i, "face": int(p is not None), "p_open_raw": p, "p_open_smoothed": tr.smoothed,
                     "ear": ear, "state": state, "closure_s": round(cl, 3), "perclos": round(tr.perclos(), 4), "event": ev, "alarm": int(alarm)})
        i += 1
    cap.release(); lmk.close()
    tr.finish(i / fps)
    return pd.DataFrame(rows), tr

video_summary = []
if RUN_VIDEO_TEST and NITYMED_DIR:
    vids = sorted(p for p in Path(NITYMED_DIR).rglob("*.mp4")
                  if "microsleep" in str(p).lower() and (not fl3d or p.stem in FL3D_TEST_VIDEOS))[:VIDEO_TEST_COUNT]
    print("Videos (FL3D test videos only):" if fl3d else "Videos:", [v.name for v in vids])
    for v in vids:
        df, tr = run_video(v, demo_config, final_model)
        df.to_csv(f"{LOGS}/video_{v.stem}_frames.csv", index=False)
        json.dump({"video": v.name, "counts": tr.counts, "events": tr.events}, open(f"{LOGS}/video_{v.stem}_events.json", "w"), indent=2)
        video_summary.append({"video": v.name, "seconds": round(df.t_s.max(), 1), "face_found": round(df.face.mean(), 3),
                              **tr.counts, "max_perclos": df.perclos.max()})
        fig, ax = plt.subplots(figsize=(14, 3))
        ax.plot(df.t_s, df.p_open_smoothed, label="P(open) smoothed"); ax.plot(df.t_s, df.perclos, label="PERCLOS")
        ax.fill_between(df.t_s, 0, 1, where=df.state == "CLOSED", alpha=0.2, color="r", label="CLOSED")
        ax.set_ylim(0, 1); ax.set_xlabel("seconds"); ax.set_title(v.name); ax.legend(loc="lower left")
        plt.savefig(f"{RESULTS}/video_{v.stem}.png", dpi=120, bbox_inches="tight"); plt.show()
    video_summary = pd.DataFrame(video_summary)
    video_summary.to_csv(f"{RESULTS}/video_test_summary.csv", index=False)
    display(video_summary)

## 13. Results summary file

In [ ]:
lines = ["# Results summary", "", f"Final model: **{FINAL_NAME}** (best mean validation accuracy).", "",
         "## Models", "",
         "| Model | Training data | Augmentation |", "|---|---|---|",
         "| v1 | our dataset | crop, flip, rotation, brightness/contrast |",
         "| v2 | our dataset | v1 + blur + low-resolution simulation |",
         "| cew_only | CEW train split | as v2 |",
         "| v3 | our dataset + CEW train split | as v2 |",
         "| v4 | our dataset + CEW train split + FL3D train drivers | as v2 |", "",
         "## Validation accuracy", "", val_table.to_markdown(floatfmt=".4f"), "",
         "## Test accuracy", "", acc.to_markdown(floatfmt=".4f"), "",
         f"## Final model on every test set (threshold {T:.2f})", "",
         results[results.model == FINAL_NAME].drop(columns="model").set_index("test_set").to_markdown(floatfmt=".4f"), "",
         "## Demo settings (config/demo_config.json)", "", "```json", json.dumps(demo_config, indent=2), "```", "",
         "## Data", "",
         f"- Our dataset: {len(split_df)} images, near-duplicates of training images in val/test: {leak}",
         f"- CEW: {len(cew)} faces with landmarks (train {len(cew_fit)}, val {len(cew_val)}, test {len(cew_test)})",
         f"- FL3D: {len(fl3d)} faces (train {len(fl3d_fit)}, val {len(fl3d_val)}, test {len(fl3d_test)})" if fl3d else "- FL3D: not available in this run",
         "", "## Speed", "", json.dumps(speed, indent=2)]
if len(video_summary):
    lines += ["", "## Real driving videos (NITYMED microsleep, FL3D test videos only)", "", video_summary.to_markdown(index=False)]
lines += ["", "## Files", ""] + [f"- `{p.relative_to(FINAL)}`" for p in sorted(Path(FINAL).rglob("*")) if p.is_file()]
Path(f"{FINAL}/README_results.md").write_text("\n".join(lines))
print("\n".join(lines[:40]))